# MICrONS PCA — Option 2: Trial-averaged PCA across cortical areas

Applying PCA per cortical area to trial-averaged responses, asking whether the three stimulus classes (Clip, Monet2, Trippy) occupy distinct regions of population state space and whether the strength of that separation differs across areas (V1, AL, LM, RL).

See `docs/specs/2026-05-02-pca-design.md` for the design.

**Sections will be filled in by subsequent tasks.**

In [ ]:
# === Validation: preprocessing functions ===
# This cell is removed in Task 9 (Setup); for now it lets us smoke-test
# the helpers against real session-7_5 data before writing the notebook proper.
import os
from pathlib import Path

import numpy as np
import microns_eda
import option2_pca_utils as pca_utils

DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
reader = microns_eda.open_dataset(DATADIR)
responses, trial_boundaries, stim_types = microns_eda.load_session_responses(
    reader, DATADIR, "7_5"
)
meta = microns_eda.get_session_meta(DATADIR, "7_5")

# Apply Stage A.
responses_pp = pca_utils.preprocess_responses(responses, apply_log=False)
assert responses_pp.shape == responses.shape, "preprocess_responses changed shape"
# After detrend + z-score, each neuron should have mean ≈ 0 and std ≈ 1.
assert np.allclose(responses_pp.mean(axis=1), 0.0, atol=1e-9), "neurons not mean-zero"
stds = responses_pp.std(axis=1, ddof=0)
assert np.all((np.abs(stds - 1.0) < 1e-9) | (stds == 0.0)), "neurons not unit-std"
print(f"preprocess_responses OK — shape {responses_pp.shape}")

# Apply Stage B.
# We need clean_trial_indices; build it from a quick treadmill scan.
# Behavior loaded as part of EDA — replicate the per-trial mean here.
per_trial_tread_means = np.empty(len(stim_types), dtype=np.float64)
for i in range(len(stim_types)):
    trial = microns_eda.load_trial(reader, DATADIR, "7_5", i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])
clean_trial_indices, _ = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=1.0
)

per_area = pca_utils.build_per_area_matrices(
    responses_pp,
    trial_boundaries=trial_boundaries,
    clean_trial_indices=clean_trial_indices,
    brain_areas=meta["brain_areas"],
    n_frames=75,
)
print(f"per_area keys: {sorted(per_area.keys())}")
for area, X in sorted(per_area.items()):
    print(f"  {area}: shape {X.shape}")
assert set(per_area.keys()) == {"V1", "AL", "LM", "RL"}, (
    f"unexpected areas: {set(per_area.keys())}"
)
assert all(X.shape[0] == len(clean_trial_indices) for X in per_area.values())
total_neurons = sum(X.shape[1] for X in per_area.values())
assert total_neurons == meta["n_neurons"], "areas don't sum to total neurons"
print(f"build_per_area_matrices OK — total {total_neurons} neurons across 4 areas")


In [ ]:
# === Validation: analysis functions on V1 (small smoke test) ===
import time

X_V1 = per_area["V1"]
# Build labels aligned to clean_trial_indices.
labels = np.array(stim_types)[clean_trial_indices]

t0 = time.time()
result = pca_utils.run_area_pipeline(
    X_V1, labels, area_name="V1",
    n_components=10,
    n_balance_replicates=20,
    n_shuffles=20,  # small for smoke test; real runs use 100
    n_folds_cv=5,
    seed=42,
)
elapsed = time.time() - t0
print(f"run_area_pipeline (V1, 20 shuffles) took {elapsed:.1f} s")

assert result["area_name"] == "V1"
assert result["n_neurons"] == X_V1.shape[1]
assert result["X_pcs"].shape == (X_V1.shape[0], 10)
assert "observed" in result["silhouette"] and "null" in result["silhouette"]
assert "observed" in result["classifier"] and "null" in result["classifier"]
print(f"  silhouette observed:    {result['silhouette']['observed']:+.3f}  (n_per_class={result['silhouette']['n_per_class']})")
print(f"  silhouette null median: {np.median(result['silhouette']['null']):+.3f}")
print(f"  silhouette p (n=20):    {result['silhouette']['empirical_p']:.3f}")
print(f"  classifier observed:    {result['classifier']['observed']:.3f}  (chance={result['classifier']['chance']:.3f})")
print(f"  classifier null median: {np.median(result['classifier']['null']):.3f}")
print(f"  classifier p (n=20):    {result['classifier']['empirical_p']:.3f}")
print("run_area_pipeline OK")